# 실습 2 — ML Planner Open-loop·Closed-loop 평가

planner 위치에 ML 모델(PLUTO)을 장착하고, 동일한 시나리오를 Open-loop 와 Closed-loop 양쪽으로
평가한다. 같은 planner 와 같은 시나리오에 대해 두 평가 방식이 서로 다른 결과를 내는 이유를
확인하는 것이 목적이다.

## 학습 목표

1. 학습된 체크포인트를 planner 로 장착한다
2. Open-loop 로 평가한다 — 로그의 주행과 얼마나 유사한가
3. Closed-loop 로 평가한다 — 이 planner 로 주행할 때 안전한가
4. 두 점수가 서로 비교 대상이 아닌 이유를 확인한다

## 실습 결과물

| 산출물 | 내용 |
|---|---|
| 영상 | 매 프레임에 스텝별 점수·자차 상태·위반 사유가 표시된다 |
| Open-loop 지표 | ADE / FDE / AHE / FHE / miss rate 와 공식 최종 점수 |
| Closed-loop 지표 | 곱셈 항 4 개와 가중 항 4 개, 총 8 개 지표와 공식 최종 점수 |
| 스텝별 CSV | 점수가 어느 시점에 하락하였는지 시간축으로 확인한다 |

## 전체 실습 구성에서의 위치

```
 실습 0            실습 1               실습 2                실습 3
 환경 구성      →  nuPlan 프레임워크  →  ML planner 평가    →  MPC refinement
 데이터 배치        구성 요소와 교체      Open-loop·Closed-loop  궤적 후처리와 비교
```

실습 1 에서 확인한 네 구성 요소 중 **planner 만 교체된다.** 나머지 세 요소는 그대로 유지한다.

## 진행 순서

| 절 | 내용 | 방식 |
|---|---|---|
| 0 | 준비 — 경로·환경변수·모델 확인 | |
| 1 | 무엇을 평가하는가 | 문서 |
| 2 | 샘플 시나리오 초기화 — 두 모델이 함께 쓴다 | API |
| 3 | **PLUTO** — 모델 구조 → 한 씬 추론 → Open-loop | API → config |
| 4 | **Diffusion Planner** — 모델 구조 → denoising 과정 → Open-loop | API → config |
| 5 | Open-loop 비교 — 두 모델 | config |
| 6 | Closed-loop 평가 — 지표 구현 → 두 모델 실행 → 비교 | config |
| 7 | 정리 | |

3 절과 4 절은 **같은 모양**이다. 모델 구조를 보고, 한 씬에 대한 입력과 추론 결과를 직접 확인한 뒤,
같은 시나리오로 Open-loop 평가를 돌린다. 바뀌는 것은 모델뿐이므로 5 절의 비교가 성립한다.

각 절 안에서 **객체를 직접 생성**하는 부분은 구조를 확인하기 위한 것이고, **Hydra 설정으로 실행**
하는 부분에서 공식 지표가 나온다. 채점과 집계가 함께 수행되는 것은 후자뿐이다.

- **커널은 `E2E Refinement`** 여야 한다.
- MPC 를 이용한 궤적 후처리는 실습 3 에서 다룬다. 본 노트북은 `use_refinement=false` 로 두므로
  **acados 를 설치하지 않았어도 진행할 수 있다.**
- 소요 시간의 대부분은 정식 시뮬레이션이며, 두 모델 × 세 프리셋(Open-loop · Closed-loop
  non-reactive · reactive)으로 **여섯 번** 돌린다. `run_sim` 은 이미 나온 결과가 있으면 건너뛰므로
  (`mode="reuse"`), 다시 열었을 때는 실행 셀을 그대로 통과한다.


## 0. 준비

devkit 이 참조할 경로와 환경변수를 설정하고, 이후 절이 필요로 하는 체크포인트·GPU·데이터가
갖추어졌는지 확인한다.


In [ ]:
# 저장소 루트를 찾고 헬퍼를 로드한 뒤, devkit 이 참조할 환경변수를 설정한다.
import sys, pathlib

NB_DIR = pathlib.Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "run_simulation.py").exists() else NB_DIR.parent
assert (REPO_ROOT / "run_simulation.py").exists(), f"저장소 루트를 찾지 못했습니다: {NB_DIR}"

helper = next(REPO_ROOT.glob("practice/**/_practice2_helper.py"))
sys.path.insert(0, str(helper.parent))

# 헬퍼가 수정되면 커널이 캐시한 옛 모듈을 계속 쓴다. 이 셀을 다시 실행하면
# 최신 내용을 읽도록 reload 를 거친다.
import importlib
import _practice2_helper
importlib.reload(_practice2_helper)
from _practice2_helper import *          # noqa: F401,F403

REPO_ROOT = bootstrap()


In [ ]:
# 사전 점검 — 체크포인트 · GPU · 로그 DB · 위젯이 모두 준비되었는지 확인한다.
import importlib.util
import warnings

# PLUTO 의 attention mask 가 bool 이 아니라는 경고. 스텝마다 나와 출력을 덮으므로 끈다.
warnings.filterwarnings("ignore", message="Converting mask without torch.bool")

ckpt = REPO_ROOT / PLUTO_CKPT
dbs = list((REPO_ROOT / "data/db/test").glob("*.db"))

section("사전 점검")
print("python  :", sys.executable)
print("ckpt    :", ckpt.name, f"({ckpt.stat().st_size/1e6:.0f} MB)" if ckpt.exists() else "없음")
print("DB      :", len(dbs), "개")
print("widgets :", "OK" if importlib.util.find_spec("ipywidgets") else "없음")

import torch
print("torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      f"| {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "")

assert ckpt.exists(), f"PLUTO 체크포인트가 없습니다: {ckpt}"

# 개수를 고정하지 않고, 시나리오 필터가 요구하는 로그가 실제로 있는지 확인한다.
import yaml
need = yaml.safe_load(
    (REPO_ROOT / "config/scenario_filter/practice_scenarios.yaml").read_text())["log_names"]
absent = [n for n in need if not (REPO_ROOT / f"data/db/test/{n}.db").exists()]
assert not absent, f"필터가 요구하는 로그 DB 가 없습니다: {absent}"
print("\n✅ 준비 완료")


GPU 가 없어도 동작하지만 스텝당 추론이 크게 느려진다. 위 출력에서 `CUDA: False` 이면
3~6 절의 정식 시뮬레이션 소요 시간이 몇 배로 늘어난다.


## 1. 무엇을 평가하는가

같은 planner 를 두 가지 방식으로 평가한다. 실습 1 §7 에서 규칙 기반 planner 로 확인한 그 구분이다.

| | Open-loop `open_loop_boxes` | Closed-loop `closed_loop_nonreactive_agents` |
|---|---|---|
| ego 의 움직임 | 로그를 그대로 재생 (`log_play_back_controller`) | 자신의 궤적으로 주행 (`two_stage_controller`) |
| 계획 궤적의 역할 | 채점 대상일 뿐, 실행되지 않는다 | 실행되어 다음 상태를 만든다 |
| 지표 | 로그 궤적과의 L2·헤딩 오차 (ADE/FDE) | 충돌·주행가능영역·진행률 등 8개 지표의 곱과 가중평균 |
| 답하는 질문 | 사람이 간 길과 얼마나 닮았는가 | 이 planner 로 주행하면 안전한가 |

두 지표는 성질이 달라 **서로 비교할 수 없다.** Open-loop 에서 오차가 작은 planner 가 Closed-loop 에서
0 점을 받을 수 있으며, 6 절에서 실제로 그런 경우를 보게 된다.

평가 대상 planner 는 `RefinementPlanner` 이다. `use_refinement=false` 로 두면 **ML 궤적을
그대로** 내보내며, 매 스텝의 채점 결과를 프레임으로 그린다. 그 프레임을 모은 것이 산출물인
영상이다.


## 2. 샘플 시나리오 초기화

3·4 절에서 두 모델의 추론을 들여다볼 **한 장면**을 만든다. 실습 1 §2 와 같은 절차이므로
헬퍼 함수로 압축한다.

planner 생성자가 살아 있는 `AbstractScenario` 를 요구하므로(채점기를 생성자 안에서 만든다)
planner 보다 시나리오가 먼저 있어야 한다.

### 필터가 둘인 이유

| 쓰이는 곳 | 필터 | 왜 |
|---|---|---|
| 3·4 절 — 모델별 한 씬 추론 | `practice_single_scenario` | 모델이 **한 장면**에서 무엇을 하는지 보는 것이라 하나면 충분하다. 두 모델이 같은 장면을 봐야 비교가 된다 |
| 3·4·6 절 — 정식 평가 | `practice_scenarios` | 점수는 시나리오마다 크게 갈리므로 **여러 개의 집계**라야 의미가 있다 |

아래 셀이 만드는 `scenario` 는 앞의 것이고, `run_sim` 이 돌리는 것은 뒤의 것이다.


In [ ]:
# 3·4 절이 들여다볼 한 장면을 만든다. 정식 평가(run_sim)는 practice_scenarios 를 따로 쓴다.
from nuplan.planning.utils.multithreading.worker_sequential import Sequential
from omegaconf import OmegaConf

SAMPLE_FILTER = "practice_single_scenario"
FILTER_YAML = OmegaConf.load(REPO_ROOT / f"config/scenario_filter/{SAMPLE_FILTER}.yaml")

builder = make_builder(REPO_ROOT)
scenarios = builder.get_scenarios(
    make_filter(scenario_tokens=list(FILTER_YAML.scenario_tokens),
                log_names=list(FILTER_YAML.log_names),
                timestamp_threshold_s=FILTER_YAML.timestamp_threshold_s),
    Sequential())

scenario = scenarios[0]
display(scenario_table(scenarios))
print("샘플:", scenario.token, "|", scenario.scenario_type)


## 3. PLUTO — 모델 구조 · 한 씬 추론 · Open-loop

<img src="assets/architecture_PLUTO_planner.png" width="880" alt="PLUTO 모델 구조">

<sub>출처: PLUTO 논문(*Push the Limit of Imitation Learning-based Planning for Autonomous Driving*) 원문 그림</sub>

PLUTO 는 장면을 토큰으로 부호화한 뒤 질의(query)로 궤적을 복원하는 Transformer 계열 모델이다.
좌측 인코더가 주변 차량(`E_A`) · 정적 객체(`E_O`) · 차선(`E_P`) · 자차(`E_AV`) 를 한 시퀀스로
통합하여 부호화하고, 우측 디코더가 횡방향·종방향 질의를 교차시켜 **여러 개의 후보 궤적과 점수**를
동시에 출력한다.

그림의 각 부분이 아래 생성자 인자에 대응한다.

| 그림 | 생성자 인자 |
|---|---|
| Transformer Encoder ×`L_enc` | `encoder_depth=4` |
| 디코더 블록 ×`L_dec` | `decoder_depth=4` |
| `Q_lat` × `Q_lon` 질의 격자 | `num_modes=12` |
| 토큰 차원 | `dim=128` |
| 출력 궤적 길이 | `future_steps=80` (0.1 s × 80 = 8 s) |
| 입력 이력 길이 | `history_steps=21` (과거 2 s) |

그림 오른쪽의 *Trajectory & Score* 는 후보가 여럿이라는 뜻이지만, 본 실습의 planner 가 읽는
것은 그중 **대표 궤적 하나**뿐이다. 화면에 보이는 점수는 그 궤적 하나를 채점한 값이다.

### 실행 구조

모델을 시뮬레이터에 장착하려면 네 개의 층이 필요하다. 각 층이 무엇을 책임지는지 확인하고 직접 만들어 본다.

```
 PlutoFeatureBuilder  →   PlanningModel   →  PlutoModelAdapter  →  RefinementPlanner
      모델 입력              추론 모델             모델 매니저              플래너
```

| 층 | 클래스 | 하는 일 |
|---|---|---|
| **모델 입력** | `PlutoFeatureBuilder` | 시뮬레이터 입력 → 모델 텐서 (반경 120 m, 주변 객체 48대, 과거 2 s) |
| **추론 모델** | `PlanningModel` | 텐서 → ego-local 궤적 `(80, 3)` |
| **모델 매니저** | `PlutoModelAdapter` | 모델마다 다른 forward 를 `AdapterOutput` 하나로 정규화하고 체크포인트를 소유 |
| **플래너** | `RefinementPlanner` | 궤적 선택 · 매 스텝 채점 · 렌더 · nuPlan `AbstractPlanner` 인터페이스 |

층이 나뉘어 있어 모델을 교체할 때 planner 를 고치지 않아도 된다. 4 절의 Diffusion Planner 는
어댑터만 바뀐다.

생성자 인자는 `config/planner/model_adapter/pluto.yaml` 의 값을 그대로 옮긴 것이며, 헬퍼의
`PLUTO_FEATURE_KWARGS` / `PLUTO_MODEL_KWARGS` 에 들어 있다.


In [ ]:
# PLUTO 를 네 층으로 직접 조립한다.
from src.feature_builders.pluto_feature_builder import PlutoFeatureBuilder
from src.models.pluto.pluto_model import PlanningModel
from src.planners.model_adapters import PlutoModelAdapter
from src.planners.refinement_planner import RefinementPlanner

SAMPLE_VIDEO_DIR = REPO_ROOT / "practice/videos/sample"

feature_builder = PlutoFeatureBuilder(**PLUTO_FEATURE_KWARGS)
model = PlanningModel(feature_builder=feature_builder, **PLUTO_MODEL_KWARGS)
adapter = PlutoModelAdapter(model, planner_ckpt=str(REPO_ROOT / PLUTO_CKPT))

planner = RefinementPlanner(
    model_adapter=adapter,
    scenario=scenario,
    use_refinement=False,               # 실습 2 는 MPC 를 쓰지 않는다
    driving_policy="ml",
    render=True,                        # 매 스텝 프레임을 모은다
    log_csv=False,                      # 커널에서는 CSV 가 누적된다 (아래 확인 사항 참조)
    render_mode="open",                 # 화면 하단을 Open-loop 지표로 그린다
    score_open_loop=True,               # 계획 궤적을 로그 ego 와 대조해 채점한다
    save_dir=str(SAMPLE_VIDEO_DIR),     # 기본값은 os.getcwd() 라 노트북 옆에 쌓인다
)
section("조립된 planner")
print("planner :", planner.name())
print("device  :", planner.device)
print("파라미터:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M")


### 체크포인트는 언제 로드되는가

생성자는 모델 구조만 만든다. **가중치는 `initialize()` 안에서 로드된다.** 어댑터의
`initialize` 가 체크포인트를 읽어 `load_state_dict` 하고 모델을 GPU 로 옮기기 때문이며,
이는 시뮬레이션이 시작될 때 한 번 일어난다.

파라미터 하나를 복사해 두고 `initialize()` 전후를 비교한다.


In [ ]:
# 생성 시점에는 가중치가 없다는 것을 확인한다 — initialize() 가 ckpt 를 로드한다.
import torch

key = [k for k in model.state_dict() if k.endswith("weight")][0]
before = model.state_dict()[key].clone()

# ego 를 로그 그대로 재생한다 — 이 절의 손 루프는 Open-loop 이다.
from nuplan.planning.simulation.controller.log_playback import LogPlaybackController

sim = make_sim(scenario, ego_controller=LogPlaybackController(scenario))
planner.initialize(sim.initialize())     # 여기서 load_state_dict + .to(device)

after = model.state_dict()[key]
section("체크포인트 로드 확인")
print("파라미터 :", key)
print("변화     :", not torch.equal(before, after.cpu()))
print("device   :", after.device)


### 같은 planner 를 Hydra 로 만들기

이 절 뒷부분의 정식 평가부터는 `run_simulation.py` 를 실행한다. 그때 planner 는 위와 같은 네 층으로 조립되지만,
인자는 yaml 에서 온다. `planner_builder` 가 `requires_scenario` 를 확인하고
`instantiate(config, scenario=scenario)` 로 시나리오를 주입하는 것이 위에서 `scenario=` 를
직접 넘긴 부분에 해당한다.


In [ ]:
# 위 네 줄과 같은 planner 를 Hydra 설정으로 만든다 — run_simulation.py 가 하는 방식이다.
# 프리셋은 planner 조립과 무관하므로 아무 것이나 넣어도 결과가 같다(여기서는 Closed-loop 프리셋).
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from nuplan.planning.script.builders.planner_builder import build_planners

GlobalHydra.instance().clear()            # 같은 프로세스에서 두 번 이상 부르려면 필요하다
with initialize_config_dir(config_dir=str(REPO_ROOT / "config")):
    cfg = compose(config_name="default_simulation", overrides=[
        "+simulation=closed_loop_nonreactive_agents",
        "planner=refinement_planner",
        "planner/model_adapter=pluto",
        "planner.refinement_planner.use_refinement=false",
        "scenario_builder=nuplan", "scenario_filter=practice_scenarios", "worker=sequential",
    ])

hydra_planner = build_planners(cfg.planner, scenario)[0]
print(type(hydra_planner).__name__, "| model_adapter:", type(hydra_planner._adapter).__name__)
del hydra_planner                          # 메모리에 두 벌을 남기지 않는다


### 빈칸 — 모델 출력을 지도 위로 옮기기

영상은 렌더러가 이미 변환을 마친 그림이다. 모델이 실제로 내놓는 것은
**ego 기준 상대좌표 `(T, 3)`** 이며, 지도 위에 그리려면 직접 옮겨야 한다.

$$
\begin{aligned}
x_g &= x_e + x_l\cos\theta - y_l\sin\theta \\
y_g &= y_e + x_l\sin\theta + y_l\cos\theta \\
\theta_g &= \mathrm{wrap}(\theta_l + \theta)
\end{aligned}
$$

회전은 **ego 의 heading $\theta$ 만큼 돌린 뒤** 위치를 더하는 순서이다. 순서를 바꾸면 안 된다.


In [ ]:
import numpy as np

def local_to_global(local, ego_x, ego_y, ego_h):
    """ego-local (T,3) [전방, 좌측, 상대헤딩] → global (T,3) [x, y, heading]."""
    local = np.asarray(local, dtype=float)
    c, s = np.cos(ego_h), np.sin(ego_h)

    # 예시 — 회전시킨 뒤 ego 위치를 더한다.
    x = ego_x + local[:, 0] * c - local[:, 1] * s

    # [TODO ①] 목적: 같은 회전을 y 성분에도 적용한다. 부호를 틀리면 궤적이
    #                지도 위에서 좌우로 뒤집혀 그려진다.
    #          채울 것: global y 좌표
    #          수식: y_g = y_e + x_l*sin(theta) + y_l*cos(theta)
    y = ...

    # [TODO ②] 목적: 각도는 더하는 것만으로 끝나지 않는다. 범위를 벗어나면
    #                이후 계산(헤딩 오차 등)이 2*pi 만큼 틀어진다.
    #          채울 것: global heading  (반드시 [-pi, pi) 로 되감을 것)
    #          수식: h_g = wrap(theta_l + theta) = arctan2(sin(theta_l+theta), cos(theta_l+theta))
    h = ...

    return np.stack([x, y, h], axis=1)


check_local_to_global(local_to_global)


채운 변환으로 **모델이 낸 점열을 직접** 그린다. 표와 영상은 점수만 보여 주지만,
여기서는 80 개 점이 실제로 어디에 놓이는지가 보인다.


In [ ]:
# 모델을 한 번 돌려 ego-local 궤적을 꺼내고, 위에서 채운 변환으로 지도 좌표에 올린다.
import matplotlib.pyplot as plt
import numpy as np

planner_input = sim.get_planner_input()
ego = planner_input.history.current_state[0]

# 어댑터를 직접 부르기 전에 planner 를 한 번 통과시킨다. RefinementPlanner 가 ScenarioManager 에
# 현재 자차 상태를 넣어 주어야 feature builder 가 경로를 찾는다 — 건너뛰면 ego_state 가 None 이라
# route_roadblock_correction 에서 AttributeError 가 난다.
planner.compute_trajectory(planner_input)

out = adapter.build_and_forward(planner_input, planner._initialization)
ml_local = out.ml_local.cpu().numpy()          # (80, 3) — ego 기준 상대좌표

mine = local_to_global(ml_local, ego.rear_axle.x, ego.rear_axle.y, ego.rear_axle.heading)
print("ml_local", ml_local.shape, "| 첫 점(local)", np.round(ml_local[0], 3))
print("변환 후 첫 점(global)", np.round(mine[0], 3))

log_xy = np.array([[s.rear_axle.x, s.rear_axle.y]
                   for s in scenario.get_expert_ego_trajectory()][:81])
plot_trajectory_points(ml_local, ego.rear_axle.x, ego.rear_axle.y,
                       ego.rear_axle.heading, log_xy=log_xy)
plt.tight_layout(); plt.show()


### 후보 궤적과 점수 — PLUTO 의 모델 특성

위 셀이 그린 것은 궤적 **하나**다. 그런데 PLUTO 는 한 번의 forward 로 후보를 **여러 개**
내놓는다. 3 절 머리말 그림의 `Q_lat × Q_lon` 질의 격자가 그것이며, reference line 마다
`num_modes=12` 개의 종방향 후보가 붙는다.

planner 가 받는 것은 그중 **점수가 가장 높은 하나**뿐이다. 모델이 `probability` 를 평탄화해
`argmax` 로 고르고, 그것이 `output_trajectory` 가 되며, 어댑터가 그것만 `ml_local` 로 꺼낸다.
후보와 점수는 어댑터에서 버려진다 — 그래서 여기서는 **모델 forward 를 직접 부른다.**

Diffusion 의 denoising 과 대비되는 지점이다. 두 모델이 궤적 하나를 내놓기까지의 과정이
전혀 다른데도, planner 코드는 한 줄도 다르지 않다.

> `probability` 는 reference line 이 패딩된 자리를 `-1e6` 으로 채워 둔다. 거르지 않으면
> 유효한 후보들의 점수 차이가 색 하나로 뭉갠다.


In [ ]:
# 후보 궤적과 점수를 상위 10개만 꺼낸다.
import matplotlib.pyplot as plt
import numpy as np

candidates, scores, pluto_data = capture_pluto_candidates(
    adapter, planner_input, planner._initialization, top_k=10)

print("후보:", candidates.shape, "= (후보, 스텝, [x y yaw])")
print("점수:", np.round(scores, 3))
print("1위와 ml_local 의 최대 차이:",
      f"{np.abs(candidates[0] - ml_local).max():.1e}")

axes = plot_pluto_candidates(candidates, scores, data=pluto_data)
axes[0].figure.suptitle("PLUTO 후보 궤적 상위 10개 — 색과 겹침 순서가 모두 학습 점수", y=1.02)
plt.show()


**확인 사항**

- **1위 후보가 곧 `ml_local` 이다.** 위 출력의 최대 차이가 `0.0e+00` 이다. 화면에서 가장 진하고
  맨 위에 그려진 궤적 하나만 planner 로 넘어가고, 나머지 아홉 개는 버려진다.
- **이 장면(`traversing_pickup_dropoff`)은 모델이 헷갈리는 장면이다.** 열 개 후보의 점수가
  `-0.97` 부터 `-2.65` 사이에 몰려 있다 — 1·2위 차이가 0.33 뿐이다. 그런데 진행 거리는
  23 m · 2.8 m · 22.8 m … 로 크게 갈린다. **비슷한 점수의 후보가 전혀 다른 행동을 뜻하는**
  상황이며, 한 스텝 뒤에 순위가 뒤집히면 궤적이 그만큼 튄다.
- **두 패널이 모두 벌어진다.** 왼쪽에서는 후보들이 도로의 굽은 결을 따라 횡방향으로 7.6 m
  퍼지고, 오른쪽에서는 8 초 진행 거리가 2.8 m 부터 24.5 m 까지 세 무리로 갈린다.
  `Q_lat × Q_lon` 격자의 **두 축이 함께 작동하는** 장면이다.
- 직진 고속 주행 장면(`high_magnitude_speed`)에서는 반대로 왼쪽이 한 줄로 겹치고 오른쪽만
  벌어진다. `SAMPLE_FILTER` 의 토큰을 바꿔 가며 어느 패널이 벌어지는지 보면, 모델이 그
  장면에서 무엇을 고민하는지 읽을 수 있다.
- **업스트림 PLUTO 는 여기서 멈추지 않았다.** 후보들을 규칙 기반으로 다시 채점해 학습 점수와
  가중합한 뒤 고르고, 그 위에 비상 제동을 얹었다. 이 실습의 `RefinementPlanner` 에는 그 층이
  **일부러 없다** — 화면의 점수 차이를 후처리 탓으로 돌릴 수 없게 하기 위해서다.


### Open-loop 파이프라인 실습

Open-loop 31 스텝을 직접 돌려, 영상의 한 프레임에 무엇이 담기는지 확인한다. 루프 본체는 실습 1 §5
의 세 줄과 같고, planner 와 `ego_controller` 가 바뀌었다.

`ego_controller` 가 `LogPlaybackController` 이므로 **자차는 planner 가 무엇을 내든 로그 그대로**
움직인다. `sim.propagate(trajectory)` 는 그대로 부르지만 controller 가 그 궤적을 쓰지 않는다 —
계획 궤적은 채점만 되고 실행되지 않는다. 아래 정식 Open-loop 평가와 같은 조건이다.

`compute_planner_trajectory` 가 아니라 `compute_trajectory` 를 부른다. 후자가 시뮬레이터가
실제로 호출하는 메서드이며, 스텝 소요 시간을 기록한다.

**31 스텝인 이유** — Open-loop 지표는 공식과 같이 **1 Hz 표본**만 누적한다. 루프는 10 Hz 로 도므로
`iteration % 10 == 0` 인 프레임, 즉 0 · 10 · 20 · 30 네 번만 누적된다. 20 스텝이면 두 번뿐이라
화면의 누적값이 거의 움직이지 않는다.

스텝당 2~3 초가 걸린다 — 추론 + 채점 + 렌더가 모두 들어 있다. GPU 가 없으면 훨씬 느리다.


In [ ]:
# Open-loop 31스텝을 직접 돌린다. 자차는 로그 재생이므로 아래 상태는 '로그의 자차' 이다.
import time

# 앞 절에서 한 씬을 추론할 때 iteration 0 이 이미 한 번 채점·렌더되었다. 지우고 시작해야
# 누적 표본이 격자 프레임(0·10·20·30) 네 개와 정확히 맞고, 이 셀을 다시 돌려도 결과가 같다.
planner._scorer.reset_open_loop()
planner._imgs.clear()

t0 = time.time()
for _ in range(31):
    planner_input = sim.get_planner_input()
    trajectory = planner.compute_trajectory(planner_input)
    sim.propagate(trajectory)
    ego = sim.history.extract_ego_state[-1]
    print(f"iter {planner_input.iteration.index:3d}  "
          f"x={ego.rear_axle.x:10.2f}  y={ego.rear_axle.y:10.2f}  "
          f"v={ego.dynamic_car_state.speed:5.2f} m/s  "
          f"a={ego.dynamic_car_state.rear_axle_acceleration_2d.x:+5.2f} m/s²")
print(f"\n{time.time()-t0:.0f}s / 31스텝")


### 프레임 한 장 확인

`render=True` 이므로 planner 가 스텝마다 프레임을 모아 두었다. 그중 한 장을 크게 띄운다.


In [ ]:
# 모아 둔 프레임 중 한 장을 크게 띄운다 — 영상의 매 프레임이 이 화면이다.
import matplotlib.pyplot as plt

print("프레임:", len(planner._imgs), "장 |", planner._imgs[0].shape)
fig, ax = plt.subplots(figsize=(11, 11))
ax.imshow(planner._imgs[-1])
ax.axis("off")
plt.tight_layout()
plt.show()


화면 요소는 다음과 같다.

`render_mode="open"` 이므로 `OpenLoopSceneRender` 가 그린다. 지도·agent·궤적은
Closed-loop 렌더러와 같고, **점수 관련 두 자리만** Open-loop 지표로 바뀐다
([openloop_scene_render.py](../src/feature_builders/openloop_scene_render.py)).

| 위치 | 내용 |
|---|---|
| 좌하단 표 | Open-loop 지표 — ADE / FDE / AHE / FHE / miss rate 와 SCORE |
| 우하단 2패널 | 계획 궤적 80 스텝의 **변위 오차와 헤딩 오차** 곡선 |
| 자홍색 실선 | ML 이 계획한 궤적 |
| 주황 상자 | 자차. 파란 상자는 주변 차량이다 |
| 회색 실선 | 주변 객체의 **로그상 미래** |

Closed-loop 렌더러(6 절)라면 이 두 자리에 충돌·주행가능영역·comfort 가 들어간다. 그 런에서
실제로 채점되는 지표를 화면에 올리는 것이라, 프리셋에 따라 화면이 달라진다.

### 영상으로 저장

모아 둔 프레임을 mp4 로 만든다. **손으로 돌린 루프에서는 이 호출을 직접 해야 한다.**
`run_simulation.py` 로 실행할 때는 `SimulationRunner` 가 시나리오 종료 시점에 불러 주지만,
여기에는 그 runner 가 없다. 한 번 부르면 모아 둔 프레임이 비워지므로 두 번 불러도 소용없다.


In [ ]:
# 모아 둔 프레임을 mp4 로 저장한다 — 손 루프에서는 이 호출을 직접 해야 한다.
planner.generate_planner_report()
saved = sorted(SAMPLE_VIDEO_DIR.glob("*.mp4"))
print(*[f"{p.name}  ({p.stat().st_size/1e6:.1f} MB)" for p in saved], sep="\n")


In [ ]:
# 저장된 mp4 를 노트북에서 재생한다.
from IPython.display import HTML

display(HTML(video_html(f"videos/sample/{saved[-1].name}")))


**확인 사항**

- **좌하단 표의 숫자는 표본 4 개짜리 중간값이다.** Open-loop 채점기는 공식과 같은 1 Hz 격자
  프레임(iteration 0 · 10 · 20 · 30)만 누적한다. 시나리오 전체를 채점한 공식 숫자는 바로 아래
  정식 평가에서 나온다.
- 이 절의 planner 는 `log_csv=False` 이므로 스텝별 채점 CSV 를 남기지 않는다. 아래 정식 런은
  `log_csv=true` 로 돌아 CSV 를 남기며, 그것을 시간축에 펼쳐 보는 것은 6 절이다.


### Open-loop 정식 평가

`run_simulation.py` 로 `practice_scenarios` 전체를 Open-loop 평가한다. 공식 지표와 집계는 이 경로로만 나온다.

`+simulation=open_loop_boxes` 프리셋은 `planner` 그룹을 `log_future_planner` 로 덮어쓴다.
따라서 `planner=refinement_planner` 를 **프리셋 뒤에** 두어야 한다. 헬퍼의 `run_sim` 이 순서를
강제하며 실행 전 전체 명령을 그대로 출력한다.

### 이 프리셋이 구성하는 네 요소

실습 1 에서 확인한 네 축이 `open_loop_boxes` 에서 어떻게 채워지는지 확인한다.

| 구성 요소 | 선택된 옵션 | 의미 |
|---|---|---|
| planner | `refinement_planner` | 우리가 지정한다. 프리셋 기본값은 `log_future_planner` 이므로 뒤에서 덮어쓴다 |
| ego_controller | **`log_play_back_controller`** | 계획 궤적을 **사용하지 않고** 로그의 자차 상태를 그대로 되돌려준다 |
| observation | `box_observation` | 주변 물체를 로그에 기록된 대로 재생한다 (`TracksObservation`) |
| simulation_time_controller | `step_simulation_time_controller` | 시나리오의 iteration 을 0 부터 끝까지 한 칸씩 진행한다 |

핵심은 `ego_controller` 다. `log_play_back_controller` 는 planner 가 무엇을 내든 자차를 로그
그대로 놓으므로, **계획 궤적은 채점만 되고 실행되지 않는다.** 자차가 planner 를 따라가지 않으니
`driving_policy` 설정도 결과에 영향을 주지 않으며, 6 절과 완전히 같은 planner 설정을 쓸 수 있다.

같은 시나리오를 4 절에서 Diffusion 으로 한 번 더 돌리며, 두 결과를 5 절에서 비교한다.

> 시나리오를 ray 워커 6개로 동시에 실행한다.


In [ ]:
# Open-loop 로 practice_scenarios 전체를 평가한다. 이미 결과가 있으면 건너뛴다
# (mode="rerun" 이면 다시 돈다). 위 한 씬 실습과 달리 여기서는 시나리오 여러 개가 필요하다.
OPEN = run_sim("open_loop_boxes", uid="practice2/pluto",
               scenario_filter="practice_scenarios", video_dir="videos")
print("결과:", OPEN)


### 최종 지표와 세부 지표

집계 parquet 에는 시나리오별 행, 시나리오 유형별 집계 행, 그리고 `final_score` 행이 섞여 있다.
`per_scenario_scores` 는 그중 시나리오별 행만 남긴다.


In [ ]:
# Open-loop 최종 점수와 시나리오별 세부 지표를 확인한다.
print(f"최종 점수: {final_score(OPEN):.4f}")
open_scores = per_scenario_scores(OPEN, columns=OPEN_BREAKDOWN)
display(open_scores.drop(columns=["video"]))


표의 다섯 개 지표는 임계로 정규화한 **점수**이다 — 앞 네 개는 `max(0, 1 - 오차/임계)` 연속값,
`miss_rate` 만 0/1 게이트이다(정의는 실습 1 §8).
실제로 몇 미터 틀렸는지는 `metrics/` 아래 원시 통계에만 있다. 3 · 5 · 8 초 지평에서의 값을 확인한다.

- **ADE** 평균 변위 오차, **FDE** 최종 변위 오차 (m) — 임계 8 m
- **AHE** 평균 헤딩 오차, **FHE** 최종 헤딩 오차 (rad) — 임계 0.8 rad
- **MR** 지평별 최대 오차가 허용치(6 · 8 · 16 m)를 넘은 표본 비율 — 0.3 을 넘으면 최종 점수가 0 이 되는 **곱셈 항**

최종 점수는 `MR 통과 여부 × (ADE + FDE + 2·AHE + 2·FHE) / 6` 이다. 헤딩 오차의 가중치가
거리 오차의 두 배이다.


In [ ]:
# 지평(3·5·8초)별 원시 오차값을 확인한다 — 집계표에는 없는 실제 오차(m, rad)이다.
horizon = openloop_horizon_table(OPEN)
display(horizon[[c for c in horizon.columns if "ADE" in c or "FDE" in c]])


### 시나리오별 영상 확인

왼쪽 목록에서 시나리오를 선택하면 해당 영상과 세부 지표가 나타난다. 점수가 낮은 순으로
정렬되어 있으므로 위쪽이 확인할 가치가 큰 장면이다.


In [ ]:
# 시나리오를 선택하면 그 시나리오의 영상과 지표가 나타난다.
display(scenario_video_browser(OPEN, tag="open_pluto", columns=OPEN_BREAKDOWN))


In [ ]:
# 실행 결과 점검 — 시나리오 수만큼 영상이 나왔는지 확인한다.
vids = collect_videos(OPEN, tag="open_pluto")
print(f"시나리오 {len(open_scores)}건 / 영상 {sum(v is not None for v in vids.values())}건")
display(runner_status(OPEN))
assert len(open_scores) > 0, "집계 parquet 이 비어 있습니다"


**확인 사항**

- 영상에서 자차(주황)와 로그 자차(회색)가 **끝까지 겹쳐 있다.** `log_play_back_controller` 가
  planner 의 궤적을 무시하고 로그를 재생하기 때문이다. 자홍색 계획 궤적만 로그에서 벗어난다.
- 화면의 점수표는 **Open-loop 지표**다. `run_sim` 이 Open-loop 런에 한해
  `score_open_loop=true` 와 `render_mode=open` 을 넣기 때문이며, 위의 손 루프에서 직접 준 인자와
  같은 것이다. 마지막 프레임의 누적 SCORE 는 위 집계표의 공식 점수와 일치한다.
- planner 를 무엇으로 바꾸어도 주행 자체는 같다. Open-loop 가 재는 것은 오직 궤적의 모양이다.


## 4. Diffusion Planner — 모델 구조 · denoising 과정 · Open-loop

<img src="assets/architecture_diffusion_planner.png" width="960" alt="Diffusion Planner 모델 구조">

<sub>출처: Diffusion Planner 논문 원문 그림</sub>

Diffusion Planner 는 궤적을 한 번에 내지 않는다. 잡음에서 시작해 **여러 번의 denoising 단계**를
거쳐 궤적을 복원하며(그림 우측 *Denoising Step*), 장면 정보는 조건으로 주입된다.

| 그림 | 하는 일 |
|---|---|
| Scenario Inputs (Neighbors · Lanes · Navigation · Static Obj) | 장면을 네 갈래로 나누어 입력한다 |
| Encoder (MLP-Mixer ×2 + MLP → Self-Attn ×N) | 네 갈래를 하나의 장면 조건으로 통합한다 |
| Diffusion Transformer | 장면 조건과 diffusion timestep 을 받아 한 단계 denoise 한다 |
| Denoising Step | 위 과정을 반복해 잡음을 궤적으로 만든다 |

PLUTO 와 구조가 전혀 다르지만 **planner 코드는 한 줄도 바뀌지 않는다.** 3 절의 네 층 중
어댑터가 모델별 forward 를 `AdapterOutput` 하나로 정규화하기 때문이며, 입력을 만드는 방식이
달라도(Diffusion 은 자체 `DataProcessor` 를 쓴다) 그 차이 또한 어댑터 안에 갇혀 있다.
바뀌는 것은 `planner/model_adapter=diffusion` 한 줄뿐이다.

이 절은 3 절과 **같은 순서**로 간다 — 모델 구조를 보고, 한 씬에 대한 입력과 추론 과정을 확인한 뒤,
같은 시나리오로 Open-loop 평가를 돌린다. 시나리오·프리셋·워커가 3 절과 모두 같으므로 5 절에서
두 결과를 그대로 비교할 수 있다.


### 한 씬 입력과 denoising 단계

3 절에서 PLUTO 는 한 번의 forward 로 궤적을 냈다. Diffusion 은 그렇지 않다 — **잡음에서
시작해 여러 단계를 거쳐** 궤적을 복원한다. 그 중간 단계를 직접 꺼내 본다.

3 절과 같은 장면(같은 시나리오의 iteration 0)을 쓴다. 다만 3 절의 `sim` 은 손 루프가 31 스텝
진행시켜 놓았으므로 새로 만든다 — `get_planner_input()` 이 돌려주는 이력은 **복사본이 아니라
살아 있는 버퍼**라서, 저장해 두어도 루프가 돌면 내용이 바뀐다.


In [ ]:
# Diffusion 을 3 절과 같은 네 층으로 조립한다. 바뀌는 것은 어댑터 한 층뿐이다.
from nuplan.planning.simulation.controller.log_playback import LogPlaybackController
from src.planners.model_adapters import DiffusionModelAdapter

DIFF_MODEL_DIR = REPO_ROOT / "data/model/diffusion_planner"

diff_adapter = DiffusionModelAdapter(
    planner_ckpt=str(DIFF_MODEL_DIR / "diffusion_planner.pth"),
    args_file=str(DIFF_MODEL_DIR / "diffusion_planner.json"),   # 구조 + 정규화 통계
    geometry_feature_builder=PlutoFeatureBuilder(**PLUTO_FEATURE_KWARGS),
    enable_ema=True,
)
diff_planner = RefinementPlanner(
    model_adapter=diff_adapter, scenario=scenario,
    use_refinement=False, driving_policy="ml", render=False, log_csv=False,
)

diff_sim = make_sim(scenario, ego_controller=LogPlaybackController(scenario))
diff_planner.initialize(diff_sim.initialize())
diff_input = diff_sim.get_planner_input()
diff_planner.compute_trajectory(diff_input)      # ScenarioManager 워밍업 — 3 절과 같은 이유

section("Diffusion planner")
print("어댑터   :", type(diff_adapter).__name__)
print("device   :", diff_planner.device)
print("파라미터 :", f"{sum(p.numel() for p in diff_adapter._planner.parameters()) / 1e6:.1f} M")


#### 모델이 보는 입력

위 그림의 *Scenario Inputs* 네 갈래가 그대로 텐서 네 개다. 어댑터는 이것을 만든 직후
`observation_normalizer` 를 통과시키므로 모델에 들어가는 값은 무차원이다. 그림으로 보려면
**정규화 전** 값이 필요해서 헬퍼가 같은 호출을 한 번 더 한다.

| 그림의 갈래 | 텐서 | 채널 |
|---|---|---|
| Neighbors | `neighbor_agents_past` (32, 21, 11) | x · y · cos · sin · vx · vy · 폭 · 길이 · 종류 onehot(3) |
| Static Obj | `static_objects` (5, 10) | x · y · cos · sin · 폭 · 길이 · 종류 onehot(4) |
| Lanes | `lanes` (70, 20, 12) | x · y · 방향(2) · 좌경계 offset(2) · 우경계 offset(2) · 신호 onehot(4) |
| Navigation | `route_lanes` (25, 20, 12) | `lanes` 와 같은 채널, 주행 경로에 속한 것만 |

**유효/패딩 마스크는 따로 없다.** 전 채널이 0 인 행이 패딩이며, 모델도 같은 규칙으로 마스크를
만든다. 좌표는 전부 **자차 뒷축 기준 미터**이고, 아래 denoising 중간 궤적과 같은 좌표계다.


In [ ]:
# Diffusion 이 실제로 보는 입력을 정규화 전 값으로 꺼내 그린다.
import matplotlib.pyplot as plt

scene = diffusion_scene_inputs(diff_adapter, diff_input)
for name, value in scene.items():
    print(f"{name:28s} {value.shape}")

fig, ax = plt.subplots(figsize=(11, 6))
plot_diffusion_scene(scene, ax=ax)
ax.set_title("Diffusion 이 보는 입력 씬 — ego-local, 정규화 전")
plt.show()


#### 중간 단계를 어떻게 꺼내는가

모델은 `dec["prediction"]` 하나만 돌려준다. 중간 `x_t` 는 밖으로 나오지 않는다. 그런데
샘플러는 이미 돌려줄 수 있게 되어 있다 — `DPM_Solver.sample(..., return_intermediate=True)`
가 단계마다 `x_t` 를 모아 주고, `dpm_sampler` 는 그 인자를 `sample_params` 로 그대로
통과시킨다. 호출부인 `decoder.py` 가 넘기지 않을 뿐이다.

그래서 **모델 코드를 고치지 않는다.** `decoder` 모듈에 `from ... import dpm_sampler` 로
바인딩된 그 **이름 하나만** 잠시 감싼다. 헬퍼의 `capture_denoising_steps` 가 하는 일이 그것이고,
지킬 것이 세 가지다.

1. 감싼 함수는 **`x0` 하나만** 돌려줘야 한다 — 호출부가 튜플을 받을 준비가 없다.
2. 모은 텐서는 **복사**해야 한다. 매 단계 현재 상태를 고정하는 `correcting_xt_fn` 이
   `x_t` 를 in-place 로 고치기 때문에, 복사하지 않으면 뒤 단계가 앞 단계를 덮어쓴다.
3. 모델이 내놓는 값은 정규화되어 있다. `decoder` 가 마지막에 하는 것과 똑같이
   `state_normalizer.inverse` 를 통과시켜야 미터가 된다.

`seed` 를 주면 `DIFFUSION_EVAL_SEED` 로 **초기 잡음이 고정**되어, 다시 돌려도 같은 그림이 나온다.
샘플러는 DPM-Solver++ 이고 단계 수는 10 으로 고정되어 있다. 여기에 워밍업과 마지막 정리 단계가
붙어 중간 텐서는 12 개가 된다.


In [ ]:
# denoising 중간 궤적을 모은다.
import numpy as np

steps, diff_out = capture_denoising_steps(
    diff_adapter, diff_input, diff_planner._initialization, seed=0)

print("중간 궤적:", steps.shape, "= (단계, 스텝, [x y yaw])")
print("마지막 단계와 AdapterOutput.ml_local 의 최대 차이:",
      f"{np.abs(steps[-1] - diff_out.ml_local.cpu().numpy()).max():.1e}")

fig, _ = plot_denoising_steps(steps, scene=scene)
fig.suptitle(f"한 씬에 대한 denoising {len(steps)} 단계", y=1.0, fontsize=12)
plt.show()


In [ ]:
# 같은 과정을 숫자로 본다 — 궤적이 언제 '궤적다워지는가'.
plot_denoising_convergence(steps)
plt.tight_layout()
plt.show()


**확인 사항**

- **마지막 단계가 곧 `AdapterOutput.ml_local` 이다.** 위 출력의 최대 차이가 `0.0e+00` 이다.
  planner 가 받는 궤적은 이 과정의 끝일 뿐, 중간 단계는 planner 에게 보이지 않는다.
- 초반 네댓 단계는 경로 길이가 1,300 m 를 넘는다. 8 초 동안 갈 수 있는 거리가 아니라
  **아직 잡음**이라는 뜻이다. 실제로 접히는 것은 5~8 단계에서 일어나고, 뒤 단계는 이미
  만들어진 궤적을 다듬는다 — 두 번째 그림의 이동량이 6 단계에서 정점을 찍고 급격히 준다.
- **잡음은 등방으로 퍼져 있는데 결과는 도로의 굽은 결을 따른다.** 8 단계부터 궤적이 휘기
  시작해 11 단계에서 차선의 곡률과 맞는다. 그 방향을 정하는 것이 `route_lanes` 이며, 위 입력
  씬 그림에서 진한 회색으로 그려진 것이 그것이다.
- 최종 경로 길이는 23.0 m 다. 같은 장면에서 PLUTO 의 1위 후보도 23.0 m 였다 — 전혀 다른
  방식으로 만든 궤적이 비슷한 결론에 도달했다. 다만 PLUTO 는 그 판단을 **후보 열 개 중
  하나를 고르는 방식**으로, Diffusion 은 **한 궤적을 열두 번 다듬는 방식**으로 했다.
- PLUTO 에는 이 과정이 없다. 한 번의 forward 로 후보 궤적들이 한꺼번에 나온다.
  **그런데 planner 코드는 두 모델에서 한 줄도 다르지 않다** — 어댑터가 이 차이를 전부
  `AdapterOutput` 안으로 감춘다.
- 이 절은 PLUTO 와 Diffusion 두 모델을 GPU 에 함께 올린다. 메모리가 모자라면 3 절의
  `planner` 와 `sim` 은 이후 절에서 쓰지 않으므로 `del` 로 정리해도 된다.


### Open-loop 정식 평가

3 절과 같은 프리셋·같은 시나리오로 돌린다. 바뀌는 것은 `planner/model_adapter=diffusion`
한 줄뿐이며, 헬퍼의 `run_sim(adapter="diffusion")` 이 그 override 를 넣는다.


In [ ]:
# Diffusion Planner 로 Open-loop 평가를 돌린다.
# 3 절의 PLUTO 런과 다른 인자는 adapter 와 uid 뿐이다 — 시나리오·프리셋·워커가 모두 같다.
DIFF_OPEN = run_sim("open_loop_boxes", uid="practice2/diffusion", adapter="diffusion",
                    scenario_filter="practice_scenarios", video_dir="videos")
print("결과:", DIFF_OPEN)


### 최종 지표와 세부 지표

> **작성 예정.** 3 절의 같은 이름 절과 대응하는 자리다. `final_score(DIFF_OPEN)` 와
> `per_scenario_scores(DIFF_OPEN, columns=OPEN_BREAKDOWN)`, `openloop_horizon_table(DIFF_OPEN)`,
> 그리고 `scenario_video_browser(DIFF_OPEN, tag="open_diffusion", columns=OPEN_BREAKDOWN)` 를
> 3 절과 같은 순서로 놓는다.


## 5. Open-loop 비교 — PLUTO 대 Diffusion

> **작성 예정.** 두 모델이 **같은 시나리오·같은 프리셋**으로 돌았으므로 여기서 처음으로 비교가
> 성립한다. 채울 내용:
>
> - 시나리오별 총점 표 — `compare_scenario_scores({"PLUTO": OPEN, "Diffusion": DIFF_OPEN},
>   columns=OPEN_BREAKDOWN)`
> - 세부 지표 비교 표와 막대그래프 — `compare_breakdown(...)` + `plot_breakdown_comparison(...)`.
>   Open-loop 에는 곱셈 항이 없으므로 그래프의 `×` 표시를 끄도록 헬퍼를 손봐야 한다.
> - 지평(3·5·8 초)별 원시 오차 대조 — `openloop_horizon_table` 두 런.
>
> 여기서 나온 순위가 6 절 Closed-loop 에서 유지되는지가 이 실습의 핵심 질문이다.


## 6. Closed-loop 평가

**두 모델을** 같은 시나리오로 Closed-loop 평가한다. 이번에는 planner 의 궤적이 실제로 실행되어
다음 상태를 만든다.

### 이 프리셋이 구성하는 네 요소

Closed-loop 에는 프리셋이 **두 개** 있다. 둘 다 이 절에서 돌린다.

| 구성 요소 | `closed_loop_nonreactive_agents` | `closed_loop_reactive_agents` |
|---|---|---|
| planner | `refinement_planner` | `refinement_planner` |
| ego_controller | `two_stage_controller` | `two_stage_controller` |
| observation | **`box_observation`** | **`idm_agents_observation`** |
| simulation_time_controller | `step_simulation_time_controller` | `step_simulation_time_controller` |

3·4 절(Open-loop)에서 바뀐 것은 `ego_controller` 하나였다. 이제 `two_stage_controller` 가
계획 궤적을 실행하므로 실습 1 §4 에서 측정한 추종 오차가 개입하고, 자차가 로그에서 점점 멀어진다.

그리고 두 Closed-loop 프리셋 사이에서 바뀌는 것은 **`observation` 하나뿐이다.**

### Non-reactive 와 Reactive 의 차이

| | Non-reactive (`box_observation`) | Reactive (`idm_agents_observation`) |
|---|---|---|
| 주변 차량 | 로그에 기록된 궤적을 그대로 재생 | IDM 으로 주행을 시뮬레이션한다 |
| 자차의 영향 | 없다 | **자차에 반응해 감속·정지한다** |
| 보행자·표지·통제 시설 | 로그 재생 | 로그 재생 (IDM 대상이 아니다) |
| 재현성 | 높다 — 주변 거동이 항상 같다 | 낮다 — 자차가 달라지면 주변도 달라진다 |

Non-reactive 에서는 자차가 앞을 막아도 주변 차량이 로그대로 지나가므로, 실제라면 뒤차가
피했을 장면이 충돌로 채점된다. Reactive 는 그 부분을 완화한다.

대신 주변 차량이 자차에 반응하므로, 같은 시나리오라도 자차가 달라지면 주변 거동도 달라진다.
지표 정의는 같지만 두 프리셋은 **서로 다른 시뮬레이션**이다.


### Closed-loop 지표 8개

최종 점수는 성격이 다른 두 부류로 구성된다.

**곱셈 항 4개** — 하나라도 0 이면 나머지와 무관하게 최종 점수가 0 이다.

| 지표 | 감점되는 상황 |
|---|---|
| `no_ego_at_fault_collisions` | 자차 과실로 충돌하였다 |
| `drivable_area_compliance` | 주행 가능 영역을 벗어났다 |
| `driving_direction_compliance` | 차선 진행 방향을 거슬러 주행하였다 |
| `ego_is_making_progress` | 경로를 따라 최소한의 전진조차 하지 못하였다 |

**가중 항 4개** — 0 과 1 사이의 값이며 가중평균한다.

| 지표 | 가중치 | 재는 것 |
|---|---|---|
| `ego_progress_along_expert_route` | 5 | 로그 자차 대비 경로 진행량의 비율 |
| `time_to_collision_within_bound` | 5 | 현재 거동을 유지할 때 충돌까지 남는 시간이 임계 이상인가 |
| `speed_limit_compliance` | 4 | 제한속도 초과량과 초과 시간 |
| `ego_is_comfortable` | 2 | 가속도 · jerk · yaw rate 가 임계 이내인가 |

```
최종 점수 = (곱셈 4개의 곱) × (5·progress + 5·ttc + 4·speed + 2·comfort) / 16
```

Open-loop 와 달리 **0 점 시나리오가 흔하다.** 충돌이나 영역 이탈이 한 번이라도 있으면 나머지
지표가 아무리 좋아도 0 이기 때문이다.


#### 빈칸 — Closed-loop 최종 점수

위 두 표의 규칙을 코드로 옮긴다. Open-loop 와 구조가 다르다 — **곱셈 항이 4 개**이고,
가중 항의 가중치가 균등하지 않다.

$$
\text{score} \;=\; \underbrace{\prod_{j} m_j}_{\text{곱셈 4개}} \;\times\;
\frac{\sum_i w_i\, m_i}{\sum_i w_i}, \qquad \sum_i w_i = 5+5+4+2 = 16
$$

곱셈 항은 0 · 0.5 · 1 처럼 이산값이고, 하나라도 0 이면 최종 점수가 0 이다.

> 이 절의 빈칸 세 개는 **채점 규칙을 먼저 이해하기 위한 것**이라 시뮬레이션 앞에 둔다.
> 채운 함수를 실제 런 결과에 적용해 공식 값과 대조하는 것은 실행이 끝난 뒤 「채운 함수를 공식
> 값과 대조」에서 한다.


In [ ]:
import numpy as np

def closed_loop_score(m):
    """m 은 지표 이름 -> 값 dict. Closed-loop 최종 점수를 돌려준다."""
    # [TODO ①] 목적: 곱셈 항의 성격을 확인한다. 안전에 해당하는 4 개는 하나라도
    #                0 이면 나머지가 아무리 좋아도 최종 점수가 0 이 되어야 한다.
    #          채울 것: 곱셈 항 4 개를 직접 곱한다
    #          수식: prod = m1 * m2 * m3 * m4
    #          어느 4 개인지는 아래 yaml 의 multiple_metrics 에 있다:
    #          nuplan/planning/script/config/simulation/metric_aggregator/
    #              closed_loop_nonreactive_agents_weighted_average.yaml
    multiplicative = (
        m["no_ego_at_fault_collisions"]
        * m["drivable_area_compliance"]
        * ...      # ego_is_making_progress
        * ...      # driving_direction_compliance
    )

    # [TODO ②] 목적: 가중치가 균등하지 않음을 확인한다. progress·TTC 는 comfort 보다
    #                2.5 배 중요하게 반영된다.
    #          채울 것: 가중 항 4 개의 가중치   (합이 16 이 되어야 한다)
    #          수식: progress 5, time_to_collision 5, speed_limit 4, comfortable 2
    #          값의 출처는 위와 같은 yaml 의 metric_weights 이다:
    #          nuplan/planning/script/config/simulation/metric_aggregator/
    #              closed_loop_nonreactive_agents_weighted_average.yaml
    w = {
        "ego_progress_along_expert_route": ...,
        "time_to_collision_within_bound": ...,
        "speed_limit_compliance": ...,
        "ego_is_comfortable": ...,
    }

    # [TODO ③] 목적: 가중 항을 하나의 값으로 합친다. 단순 평균이 아니라 가중평균이다.
    #          채울 것: 분자(가중합)와 분모(가중치 합)를 직접 더한다
    #          수식: weighted = (w1*m1 + w2*m2 + w3*m3 + w4*m4) / (w1 + w2 + w3 + w4)
    numerator = (
        w["ego_progress_along_expert_route"] * m["ego_progress_along_expert_route"]
        + w["time_to_collision_within_bound"] * m["time_to_collision_within_bound"]
        + ...      # speed_limit_compliance
        + ...      # ego_is_comfortable
    )
    denominator = (
        w["ego_progress_along_expert_route"]
        + w["time_to_collision_within_bound"]
        + ...      # speed_limit_compliance
        + ...      # ego_is_comfortable
    )
    weighted = numerator / denominator

    return multiplicative * weighted


check_closed_loop_score(closed_loop_score)


#### 빈칸 — 개별 지표 두 개

위 빈칸은 8 개 지표를 **합치는** 규칙이었다. 지표 자체가 어떻게 만들어지는지도 본다.
8 개 중 연속값이 나오는 것은 사실상 `progress` 와 `speed_limit` 둘뿐이고, 나머지는 0/1 이다.

**① 진행률** — expert 대비 얼마나 나아갔는가

$$
\text{ratio} = \min\!\left(1,\ \frac{\max(p_{ego},\ \tau)}{\max(p_{expert},\ \tau)}\right),
\qquad \tau = 2\,\mathrm{m}
$$

분자·분모에 모두 하한 $\tau$ 를 두는 이유는 **expert 가 거의 정지한 시나리오**(신호 대기 등)
때문이다. 하한이 없으면 0 으로 나누거나, 조금만 움직여도 비율이 폭발한다.
다만 ego 가 $-\tau$ 보다 더 **뒤로** 갔다면 비율을 따지지 않고 0 이다.

**② 제한속도 준수** — 초과의 크기와 지속시간을 함께 본다

$$
\text{loss} = \frac{\sum_k v^{over}_k \cdot \Delta t}{v_{max}\cdot T},
\qquad \text{score} = \max(0,\ 1 - \text{loss}),
\qquad v_{max} = 2.23\,\mathrm{m/s}
$$

분자는 초과량의 **면적**(m/s × s)이다. 잠깐 크게 넘긴 것과 오래 조금 넘긴 것이 같게 평가된다.


In [ ]:
def progress_score(ego_progress, expert_progress, threshold=2.0):
    """expert 대비 진행률. threshold 는 devkit 의 score_progress_threshold (2 m)."""
    # [TODO ①] 목적: 뒤로 간 주행은 비율로 환산하지 않고 곧바로 0 으로 만든다.
    #                (데이터 잡음으로 -threshold 까지는 허용한다)
    #          채울 것: 조건식 — ego 가 임계보다 더 뒤로 갔는가
    #          수식: ego_progress < -threshold
    if ...:
        return 0.0

    # [TODO ②] 목적: 분자·분모에 하한을 두어 expert 가 거의 정지한 시나리오에서
    #                0 나눗셈과 비율 폭발을 막는다.
    #          채울 것: 진행률 ratio  ([0, 1] 로 saturate)
    #          수식: ratio = min(1, max(ego, threshold) / max(expert, threshold))
    #          출처: nuplan/planning/script/config/common/simulation_metric/low_level/
    #                    ego_progress_along_expert_route_statistics.yaml
    ratio = ...
    return ratio


check_progress_score(progress_score)


In [ ]:
def speed_limit_score(overspeed, dt, duration, max_overspeed=2.23):
    """overspeed 는 스텝별 초과 속도[m/s] 목록. 초과가 없으면 0 이 들어 있다."""
    # [TODO ①] 목적: 위반을 '한 번이라도 넘었는가' 가 아니라 크기 × 시간의 면적으로 잰다.
    #          채울 것: 위반 손실 violation_loss
    #          수식: loss = sum(overspeed) * dt / (max_overspeed * duration)
    #          출처: nuplan/planning/script/config/common/simulation_metric/high_level/
    #                    speed_limit_compliance_statistics.yaml   (2.23 m/s)
    violation_loss = ...

    # [TODO ②] 목적: 손실이 1 을 넘어도 점수가 음수가 되면 안 된다.
    #          채울 것: 점수 score
    #          수식: score = max(0, 1 - loss)
    score = ...
    return score


check_speed_limit_score(speed_limit_score)


### 두 모델 Closed-loop 실행

3·4 절의 Open-loop 와 같은 시나리오를, 이번에는 `two_stage_controller` 로 돌린다. 계획 궤적이
실제로 실행되므로 자차가 로그에서 점점 멀어진다.


In [ ]:
# PLUTO 를 Closed-loop 로 평가한다 — 3 절 Open-loop 과 같은 시나리오다.
CLOSED = run_sim("closed_loop_nonreactive_agents", uid="practice2/pluto",
                 scenario_filter="practice_scenarios", video_dir="videos")
print("결과:", CLOSED)


In [ ]:
# 두 모델을 Closed-loop 로 평가한다. 위 PLUTO 런과 adapter·uid 만 다르다.
DIFF_CLOSED = run_sim("closed_loop_nonreactive_agents", uid="practice2/diffusion",
                      adapter="diffusion", scenario_filter="practice_scenarios",
                      video_dir="videos")
print("결과:", DIFF_CLOSED)


### 채운 함수를 공식 값과 대조

채운 함수를 **실제 런의 시나리오별 8 지표**에 적용해 nuPlan 이 계산한 점수와 대조한다.


In [ ]:
# 시나리오별 8지표를 내 함수에 넣고 공식 점수와 비교한다.
cl = closed_loop_inputs(CLOSED)
cl["mine"] = [closed_loop_score(r._asdict()) for r in cl.itertuples()]
cl["차이"] = (cl["mine"] - cl["official"]).abs()
display(cl.round(4))
print("최대 차이:", f"{cl['차이'].max():.2e}")


진행률은 런의 원시 진행량이 parquet 에 남아 있어 **공식 값과 직접 대조**할 수 있다.
(제한속도는 초과 속도의 시계열이 남지 않아 위 검사로만 확인한다.)


In [ ]:
# ego/expert 진행량을 내 함수에 넣고 공식 비율과 비교한다.
pg = progress_inputs(CLOSED)
pg["mine"] = [progress_score(r.ego_progress, r.expert_progress) for r in pg.itertuples()]
pg["차이"] = (pg["mine"] - pg["official"]).abs()
display(pg.round(4))
print("최대 차이:", f"{pg['차이'].max():.2e}")


In [ ]:
# Closed-loop 최종 점수와 시나리오별 세부 지표를 확인한다.
print(f"최종 점수: {final_score(CLOSED):.4f}")
closed_scores = per_scenario_scores(CLOSED, columns=CLOSED_BREAKDOWN)
display(closed_scores.drop(columns=["video"]))


### 스텝별 점수 추이

planner 가 매 스텝 남긴 채점 결과로 점수가 언제 무너졌는지 확인한다. 영상의 점수표를 시간축에
펼친 것과 같다.


In [ ]:
# 스텝별 채점 결과를 시간축에 펼쳐 언제 점수가 무너졌는지 확인한다.
import matplotlib.pyplot as plt

steps = load_step_csv(CLOSED)
if steps is None:
    print("스텝별 CSV 가 없습니다.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    for token, g in steps.groupby("token"):
        g = g.sort_values("iteration")
        ax.plot(g["iteration"] * 0.1, g["ml_final"], lw=1.6, label=token[:8])
    ax.set_xlabel("time [s]"); ax.set_ylabel("스텝 점수")
    ax.set_ylim(-0.05, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=3)
    ax.set_title("Closed-loop 스텝별 점수")
    plt.show()


### 시나리오별 영상 확인

두 런의 영상을 각각 연다. 같은 토큰을 양쪽에서 열어 같은 장면을 비교하는 것이 목적이다.


In [ ]:
# 시나리오를 선택하면 그 시나리오의 영상과 지표가 나타난다.
display(scenario_video_browser(CLOSED, tag="closed_pluto", columns=CLOSED_BREAKDOWN))


In [ ]:
# 차이가 큰 시나리오를 골라 그 토큰의 영상을 확인한다.
display(scenario_video_browser(DIFF_CLOSED, tag="closed_diffusion", columns=CLOSED_BREAKDOWN))


### Closed-loop 점수 비교 — PLUTO 대 Diffusion

두 런은 `planner/model_adapter` 한 줄만 다르다. 시나리오 필터·프리셋·컨트롤러·관측이 모두
같으므로, 점수 차이는 **모델 차이**로 읽을 수 있다. 비교 함수는 그 전제를 코드로도 확인한다 —
`(log_name, token)` 이 양쪽에 다 있는 시나리오만 남기고, 빠진 것이 있으면 몇 건인지 출력한다.
한쪽 시나리오가 실패해 표본이 어긋나면 평균이 서로 다른 표본의 평균이 되어, 비교 자체가
성립하지 않기 때문이다.

Open-loop 비교는 5 절에서 이미 했다. 두 지표는 다른 성질을 재므로 한 표에 섞지 않는다 —
순위가 뒤집히는 것은 이 절 끝에서 따로 확인한다.

총점은 두 가지가 나온다.

| | 무엇인가 |
|---|---|
| 공식 최종 점수 | nuPlan 집계 규칙 — 시나리오 **유형**별로 평균한 뒤 그 평균들을 다시 평균한다 |
| `score` 행 | **공통 시나리오**의 단순 평균 |

`practice_scenarios` 는 유형이 겹치지 않으므로 두 값은 같다. 갈라진다면 표본이 어긋난
것이며, 그때 비교의 근거가 되는 것은 `score` 행이다.


In [ ]:
# Closed-loop 총점을 나란히 놓는다. dict 에 적은 순서가 그대로 표의 열 순서다.
RUNS = {"PLUTO": CLOSED, "Diffusion": DIFF_CLOSED}

for name, run in RUNS.items():
    print(f"{name:10s} 공식 최종 점수 {final_score(run):.4f}")

display(compare_scenario_scores(RUNS))


In [ ]:
# 세부 지표를 공통 시나리오 평균으로 비교한다 — 총점 차이가 어느 항에서 왔는지 본다.
import matplotlib.pyplot as plt

breakdown = compare_breakdown(RUNS)
display(breakdown.round(3))

plot_breakdown_comparison(breakdown, title="Closed-loop 세부 지표 — PLUTO 대 Diffusion")
plt.show()


**확인 사항**

- **총점이 비슷해도 세부 지표는 전혀 다른 모양일 수 있다.** 막대그래프에서 `×` 가 붙은 네 항은
  곱셈 항이라, 낮다는 것은 "평균적으로 조금 못한다" 가 아니라 **몇몇 시나리오에서 0 점**을
  받았다는 뜻이다. 시나리오별 표에서 그 시나리오가 어느 것인지 바로 확인된다.
- 곱셈 항이 아닌 네 항은 가중합이며, 여기서 벌어진 차이는 총점에 완만하게 반영된다
  (progress 5 · TTC 5 · speed limit 4 · comfort 2).
- 시나리오별 표의 `차이` 가 큰 토큰을 영상 브라우저에서 열면, 숫자가 갈린 지점을 눈으로
  확인할 수 있다. 두 모델이 같은 장면에서 서로 다른 판단을 한 순간이다.
- `practice_scenarios` 는 시나리오 수가 적으므로 이 비교는 **경향을 보는 것**이지
  모델의 우열을 정하는 것이 아니다.
  Closed-loop 은 곱셈 항 하나로 시나리오 점수가 0 이 되므로, 표본이 적으면 시나리오 한 건이
  총점을 좌우한다.


### Reactive 로 다시 평가

`observation` 만 바꾸어 같은 planner·같은 시나리오를 다시 돌린다. 주변 차량이 IDM 으로
움직이므로 Non-reactive 보다 다소 오래 걸린다. **두 모델 모두** 돌려야 비교가 성립한다.


In [ ]:
# observation 만 다른 프리셋으로 같은 평가를 반복한다 — 두 모델 모두.
REACTIVE = run_sim("closed_loop_reactive_agents", uid="practice2/pluto",
                   scenario_filter="practice_scenarios", video_dir="videos")
DIFF_REACTIVE = run_sim("closed_loop_reactive_agents", uid="practice2/diffusion",
                        adapter="diffusion", scenario_filter="practice_scenarios",
                        video_dir="videos")
print("결과:", REACTIVE, DIFF_REACTIVE, sep="\n  ")


In [ ]:
# 프리셋별로 따로 출력한다. 하나의 표로 합치지 않는다 — 서로 다른 시뮬레이션이다.
for label, runs in [("non-reactive", {"PLUTO": CLOSED, "Diffusion": DIFF_CLOSED}),
                    ("reactive", {"PLUTO": REACTIVE, "Diffusion": DIFF_REACTIVE})]:
    section(f"{label}   " + "   ".join(f"{n} {final_score(r):.4f}" for n, r in runs.items()))
    display(compare_scenario_scores(runs))


**확인 사항**

- **비교는 표 안에서만 한다.** 같은 프리셋 안의 두 모델은 완전히 같은 조건이므로 비교가
  성립하고, 그래서 위 출력이 프리셋마다 한 표씩이다. 반대로 프리셋을 가로질러 빼는 것은
  안 된다 — 같은 시나리오라도 주변 차량이 다르게 움직여 자차가 놓인 상황 자체가 다르고, 점수
  차이를 무엇에 귀속시킬지 알 수 없다. nuPlan 챌린지도 두 부문을 **별개 항목**으로 둔다.
- 각 표 안에서 읽을 것은 같다 — 곱셈 항이 0 인 시나리오와 그 원인이다.
- 방향은 한쪽으로 정해지지 않는다. Reactive 에서 주변 차량이 양보해 상황이 쉬워지는
  시나리오도 있고, IDM 차량이 자차 앞에서 감속해 진행이 막히거나 새로운 충돌이 생기는
  시나리오도 있다. 두 프리셋 중 어느 쪽이 옳은 평가인지는 정해져 있지 않다 —
  Non-reactive 는 재현성이, Reactive 는 주변 반응의 현실성이 강점이다.


In [ ]:
# Open-loop 와 Closed-loop 점수를 나란히 놓는다 — 순위가 뒤집히는지 확인한다.
import pandas as pd

cmp = pd.merge(
    open_scores[["token", "scenario_type", "score"]].rename(columns={"score": "open_loop"}),
    closed_scores[["token", "score"]].rename(columns={"score": "closed_loop"}),
    on="token")
display(cmp.sort_values("closed_loop"))


**확인 사항**

- **Open-loop 상위 시나리오가 Closed-loop 에서 0 점이 될 수 있다.** 순위가 단순히 흔들리는 정도가 아니라
  뒤집힌다. 로그와 닮은 궤적을 냈다는 것이 그 궤적으로 주행해도 안전하다는 뜻은 아니다.
- Closed-loop 0 점은 곱셈 항 하나가 0 이 된 결과이며, 원인은 시나리오마다 다르다. 세부 지표 표에서
  어느 항이 0 인지 확인한 뒤 영상의 해당 시점에서 위반 사유 상자를 보면 무슨 일이 있었는지
  읽을 수 있다. `no_ego_at_fault_collisions=0` 이면 충돌이고,
  `ego_is_making_progress=0` 이면 자차가 거의 나아가지 못한 것이다.
- **프레임의 점수표와 공식 점수는 같은 것이 아니다.** 스텝 점수가 내내 1.000 인데 공식 점수가
  0 인 시나리오가 나올 수 있다. 프레임의 채점기는 진행률 계열을 1.0 으로 고정하기 때문이다 —
  Closed-loop 에서는 자차가 로그와 발산하여 진행률의 분모가 시나리오마다 다른 의미를 갖게 되고,
  그러면 스텝 간 비교가 성립하지 않는다. 진행 실패는 공식 집계에서만 드러난다.
- Closed-loop 에서는 스텝이 지날수록 자차가 로그에서 멀어진다. 실습 1 §4 에서 확인한 추종 오차와,
  planner 가 매 스텝 새로 계획한 결과가 함께 쌓인 것이다.


## 7. 정리

두 모델을 같은 시나리오에 적용했고, 프리셋 하나로 두 개의 점수 체계가 나왔다.

| | Open-loop | Closed-loop |
|---|---|---|
| ego_controller | `log_play_back_controller` | `two_stage_controller` |
| 계획 궤적 | 채점만 된다 | 실행되어 다음 상태를 만든다 |
| 재는 것 | 궤적이 로그와 얼마나 닮았는가 | 그 궤적으로 주행하면 안전한가 |
| 한계 | 실행되지 않으므로 오차 누적이 드러나지 않는다 | 시나리오마다 결과가 갈려 표본이 많이 필요하다 |

기억할 것 세 가지이다.

1. **ML planner 도 네 구성 요소 중 planner 자리 하나일 뿐이다.** 나머지 세 축을 고정해야
   모델끼리 비교가 성립한다 — 3 절과 4 절이 `model_adapter` 한 줄만 다른 이유다.
2. **Open-loop 점수는 Closed-loop 성능을 보장하지 않는다.** 두 지표는 다른 성질을 잰다.
3. **곱셈 항 하나가 전체를 지배한다.** 평균적으로 좋은 planner 라도 드물게 발생하는 위반이
   최종 점수를 0 으로 만든다.

### 다음 — 실습 3

여기서 확인한 ML 궤적을 **저역통과 필터와 MPC 로 후처리**하여, 후처리 전후를 같은 조건에서
매 스텝 채점해 비교한다. 영상에는 세 궤적과 세 열의 점수표가 함께 표시된다.

[practice3_ml_planner_refinement.ipynb](practice3_ml_planner_refinement.ipynb) 로 이어진다.

필터 절(§2~§4)은 acados 없이 진행되지만, MPC 절(§5~§8)은 실습 0(**Day 1**)의 6·7 단계가
끝나 있어야 한다. 그 사이 이틀이 비므로, **오늘 안에 아래 한 줄을 돌려 두면** 실습 3 에서
막히지 않는다. 실패하면 `bash script/install_acados.sh && bash script/build_mpc.sh` 다.

```bash
source script/nuplan_env.sh && \
python -c "from src.planners.utils.mpc_interface import RefinementMpcInterface; \
RefinementMpcInterface(); print('MPC 로드 OK')"
```
